In [1]:
from numpy.random import default_rng
import scanpy as sc
# import squidpy as sq
from anndata import AnnData
import scipy
from sklearn.metrics.cluster import adjusted_rand_score
import numpy as np
import pandas as pd
import seaborn as sns
import os
import torch
import pandas as pd
from sklearn import metrics
import multiprocessing as mp
from sklearn.metrics.cluster import normalized_mutual_info_score
import pickle
from sklearn.neighbors import kneighbors_graph
from sklearn.mixture import GaussianMixture
from sklearn.mixture import BayesianGaussianMixture
import time
from utils import cKD_refine_label
sc.logging.print_header()

/home/vanhoan310/miniconda3/envs/gnn/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


scanpy==1.9.3 anndata==0.9.2 umap==0.5.7 numpy==1.24.4 scipy==1.8.1 pandas==1.4.2 scikit-learn==1.3.2 statsmodels==0.14.1 python-igraph==0.11.8 pynndescent==0.5.13


In [2]:
from sklearn.cluster import (KMeans, MiniBatchKMeans, AgglomerativeClustering, DBSCAN, OPTICS, MeanShift, SpectralClustering, Birch)
from sklearn.mixture import GaussianMixture, BayesianGaussianMixture
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score, homogeneity_score, completeness_score

In [3]:
## Parallel implementation
from joblib import Parallel, delayed
def process_slice_donor(batch_id):
    # print(sliceID, donorID)
    adata = adata_big[adata_big.obs['batch'] == batch_id].copy()
    if adata.shape[0] > 10:
        n_clusters = 200 #100       
        # Clustering
        connectivity = kneighbors_graph(adata.obsm['spatial'], n_neighbors=4, include_self=False)
        connectivity = 0.5 * (connectivity + connectivity.T)
        embedding = connectivity.dot(adata.obsm['X_pca'])

        clusteringobj = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, tol=1e-3, max_iter=10000)
        adata.obs[GraphBGM] = clusteringobj.fit_predict(embedding)
        # adata.obs[GraphBGM] = refine_label(adata, radius=50, key=GraphBGM)
        adata.obs[GraphBGM] = cKD_refine_label(np.array(adata.obsm['spatial']), adata.obs[GraphBGM], k = 45)
        cluster_vec = np.unique(adata.obs[GraphBGM])
        clusterIDvec = []
        clusterValuevec = []
        cluster2Cellsvec = []
        for clusterID in cluster_vec:
        # for clusterID in range(n_clusters):
            sliceClusterID = batch_id + 'ClusterID' + str(clusterID)
            sliceClusterValue = np.mean(embedding[adata.obs[GraphBGM] == clusterID], axis=0)
            clusterIDvec.append(sliceClusterID)
            clusterValuevec.append(sliceClusterValue)
            cluster2Cellsvec.append(adata.obs_names[adata.obs[GraphBGM] == clusterID])
        return ((clusterIDvec, clusterValuevec, cluster2Cellsvec))

In [4]:
# MERFISH data
data_dir = '/data/hoan/spatial_transcriptomics/data/SDMBench/Data/378k/BrainAgingSpatialAtlas_MERFISH.h5ad'
adata_full=sc.read_h5ad(data_dir)
donor_id = np.unique(adata_full.obs['donor_id'])
adata_full.obsm['spatial'] = adata_full.obsm['spatial_coords']
adata_full.obs['batch'] = adata_full.obs['donor_id'].astype(str) +'Slice'+ adata_full.obs['slice'].astype(str)
adata_full.obs['Region'] = adata_full.obs['tissue']

In [5]:
GraphBGM = 'domain'#'GraphBGMv3a2full'
adata_full.obs['pseudoClustLabel'] = 0

In [6]:
# from utils import clustering_v3a2
from sklearn.cluster import (KMeans, MiniBatchKMeans, AgglomerativeClustering, SpectralClustering, Birch)
from sklearn.neighbors import kneighbors_graph
from GraphST.utils import refine_label

In [7]:
n_clusters = 10

In [8]:
import numpy as np

batch_numbers = [5, 10, 15, 20, 25, 30]

# Get unique batch IDs
batch_ids = adata_full.obs['batch'].unique()

rng = np.random.default_rng(42)

for n_batches in batch_numbers:

    # Randomly select n batch IDs
    selected_batches = rng.choice(
        batch_ids,
        size=n_batches,
        replace=False
    )

    # Subset cells belonging to these batches
    adata_big = adata_full[
        adata_full.obs['batch'].isin(selected_batches)
    ].copy()

    print(
        f"{n_batches} batches: "
        f"{adata_big.n_obs} cells"
    )

    time_st = time.time()
    # Parallel execution
    batch_vec = np.unique(adata_big.obs['batch'])
    # batch_vec = batch_vec[:5]
    results = Parallel(n_jobs=10)(delayed(process_slice_donor)(batch) for batch in batch_vec)
    # time_ed = time.time()
    # time_batch_clustering = time_ed-time_st
    # print(time_batch_clustering/60)
    
    print(batch_vec)
    
    sliceCluster2SampleID = {}
    pseudoSamples_column = []
    pseudoSamples_value = []
    for i in range(len(results)):
        if results[i] is not None:
            x, y, z = results[i]
            pseudoSamples_column += x
            pseudoSamples_value += y
            for j in range(len(x)):
                sliceCluster2SampleID[x[j]] = z[j]
    
    # pseudoSamples = pd.DataFrame(np.array(pseudoSamples_value)[:len(pseudoSamples_column),].T, columns = list(pseudoSamples_column))
    pseudoSamples = pd.DataFrame(np.array(pseudoSamples_value).T, columns = list(pseudoSamples_column))
    n_clusters = len(set(adata_big.obs['Region']))
    if np.nan in set(adata_big.obs['Region']):
        n_clusters = n_clusters-1 # remove nan
    
    # Define clustering algorithms
    clustering_algorithms = {
        'GraphBG': BayesianGaussianMixture(n_components=n_clusters, covariance_type='tied', random_state=42, init_params = 'random_from_data', n_init = 15, max_iter = 1000),
    }
    pseudoSamples_list = pseudoSamples_column
    if 'BrainAging' in pseudoSamples_list[0]:
        # print('Set batch for Brain Aging Dataset')
        batch_labels = [item[0:30] for item in pseudoSamples_list] # For 
    else:
        print('Set batch for Maynard dataset')
        batch_labels = [item[0:5] for item in pseudoSamples_list] # For
    # print(np.unique(batch_labels))
    X = pseudoSamples.T.values
    import scanpy.external as sce
    adata = sc.AnnData(X)
    adata.obsm['X_pca'] = X
    adata.obs["batch"] = batch_labels
    # time_st = time.time()
    # sc.pp.combat(adata, key="batch", inplace=True)
    # sce.pp.scanorama_integrate(adata, key="batch")
    sce.pp.harmony_integrate(adata, key="batch", max_iter_harmony=40)
    # time_ed = time.time()
    # time_combat = time_ed-time_st
    # adata
    X = adata.X
    if 'X_pca_harmony' in adata.obsm:
        print ("Use harmony")
        X  = adata.obsm['X_pca_harmony']
    # Store clustering results
    # time_st = time.time()
    results = {}
    for name, model in clustering_algorithms.items():
        if hasattr(model, 'fit_predict'):
            labels = model.fit_predict(X)
        else:
            model.fit(X)
            labels = model.predict(X)
        # Store labels for visualization
        results[name] = labels
        multi_clusterLabels = labels
        adata_big.obs['pseudoClustLabel'] = 0
        for i in range(len(pseudoSamples_list)):
            pseudoSamplesX = pseudoSamples_list[i]
            adata_big.obs.loc[sliceCluster2SampleID[pseudoSamplesX], 'pseudoClustLabel'] = multi_clusterLabels[i]
        #print(name, normalized_mutual_info_score(adata_big.obs['tissue'], adata_big.obs['pseudoClustLabel']))
        print('Spatial correction per slice')
        for batch in batch_vec:
            mask_ = (adata_big.obs['batch'] == batch)
            adata_big.obs.loc[mask_, 'pseudoClustLabel'] = cKD_refine_label(np.array(adata_big[mask_].obsm['spatial']),
                                                                            adata_big.obs.loc[mask_, 'pseudoClustLabel'], k = 45)
        true_labels = pd.factorize(adata_big.obs['Region'])[0]
        print(name, normalized_mutual_info_score(true_labels[true_labels>0], adata_big.obs['pseudoClustLabel'][true_labels>0]))
    
    time_ed = time.time()
    time_total = time_ed-time_st
    
    print("Total time in minutes: ", time_total/60.0)
    # print("Total time in minutes: ", (time_batch_clustering+time_combat+time_Kmeans)/60.0)

    print('============================================================================================')

5 batches: 66079 cells


2026-09-09 09:37:46,645 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...


['MsBrainAgingSpatialDonor_12Slice0' 'MsBrainAgingSpatialDonor_2Slice0'
 'MsBrainAgingSpatialDonor_6Slice2' 'MsBrainAgingSpatialDonor_8Slice1'
 'MsBrainAgingSpatialDonor_9Slice1']


2026-09-09 09:37:47,649 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-09-09 09:37:47,656 - harmonypy - INFO - Iteration 1 of 40
2026-09-09 09:37:47,874 - harmonypy - INFO - Iteration 2 of 40
2026-09-09 09:37:48,057 - harmonypy - INFO - Iteration 3 of 40
2026-09-09 09:37:48,242 - harmonypy - INFO - Iteration 4 of 40
2026-09-09 09:37:48,431 - harmonypy - INFO - Iteration 5 of 40
2026-09-09 09:37:48,826 - harmonypy - INFO - Iteration 6 of 40
2026-09-09 09:37:49,057 - harmonypy - INFO - Iteration 7 of 40
2026-09-09 09:37:49,169 - harmonypy - INFO - Iteration 8 of 40
2026-09-09 09:37:49,351 - harmonypy - INFO - Iteration 9 of 40
2026-09-09 09:37:49,479 - harmonypy - INFO - Iteration 10 of 40
2026-09-09 09:37:49,646 - harmonypy - INFO - Iteration 11 of 40
2026-09-09 09:37:49,752 - harmonypy - INFO - Iteration 12 of 40
2026-09-09 09:37:49,839 - harmonypy - INFO - Iteration 13 of 40
2026-09-09 09:37:49,929 - harmonypy - INFO - Iteration 14 of 40
2026-09-09 09:37:50,030 - ha

Use harmony
Spatial correction per slice
GraphBG 0.667477392543116
Total time in minutes:  0.6192325274149577
10 batches: 121208 cells


...

Use harmony
Spatial correction per slice
GraphBG 0.6753261790418431
Total time in minutes:  2.526734761397044
30 batches: 363180 cells


2026-09-09 09:45:52,804 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...


['MsBrainAgingSpatialDonor_10Slice0' 'MsBrainAgingSpatialDonor_10Slice1'
 'MsBrainAgingSpatialDonor_10Slice2' 'MsBrainAgingSpatialDonor_11Slice0'
 'MsBrainAgingSpatialDonor_11Slice1' 'MsBrainAgingSpatialDonor_11Slice2'
 'MsBrainAgingSpatialDonor_12Slice0' 'MsBrainAgingSpatialDonor_1Slice0'
 'MsBrainAgingSpatialDonor_2Slice0' 'MsBrainAgingSpatialDonor_2Slice1'
 'MsBrainAgingSpatialDonor_3Slice0' 'MsBrainAgingSpatialDonor_3Slice1'
 'MsBrainAgingSpatialDonor_4Slice0' 'MsBrainAgingSpatialDonor_4Slice1'
 'MsBrainAgingSpatialDonor_4Slice2' 'MsBrainAgingSpatialDonor_5Slice0'
 'MsBrainAgingSpatialDonor_5Slice1' 'MsBrainAgingSpatialDonor_5Slice2'
 'MsBrainAgingSpatialDonor_6Slice0' 'MsBrainAgingSpatialDonor_6Slice1'
 'MsBrainAgingSpatialDonor_6Slice2' 'MsBrainAgingSpatialDonor_7Slice0'
 'MsBrainAgingSpatialDonor_7Slice1' 'MsBrainAgingSpatialDonor_7Slice2'
 'MsBrainAgingSpatialDonor_8Slice0' 'MsBrainAgingSpatialDonor_8Slice1'
 'MsBrainAgingSpatialDonor_8Slice2' 'MsBrainAgingSpatialDonor_9Slice0'

2026-09-09 09:45:55,024 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-09-09 09:45:55,091 - harmonypy - INFO - Iteration 1 of 40
2026-09-09 09:45:57,109 - harmonypy - INFO - Iteration 2 of 40
2026-09-09 09:45:59,085 - harmonypy - INFO - Iteration 3 of 40
2026-09-09 09:46:01,080 - harmonypy - INFO - Iteration 4 of 40
2026-09-09 09:46:03,088 - harmonypy - INFO - Iteration 5 of 40
2026-09-09 09:46:05,344 - harmonypy - INFO - Iteration 6 of 40
2026-09-09 09:46:07,329 - harmonypy - INFO - Iteration 7 of 40
2026-09-09 09:46:09,308 - harmonypy - INFO - Iteration 8 of 40
2026-09-09 09:46:11,297 - harmonypy - INFO - Iteration 9 of 40
2026-09-09 09:46:12,502 - harmonypy - INFO - Iteration 10 of 40
2026-09-09 09:46:13,333 - harmonypy - INFO - Iteration 11 of 40
2026-09-09 09:46:14,131 - harmonypy - INFO - Iteration 12 of 40
2026-09-09 09:46:14,871 - harmonypy - INFO - Iteration 13 of 40
2026-09-09 09:46:15,667 - harmonypy - INFO - Iteration 14 of 40
2026-09-09 09:46:16,371 - ha

Use harmony
Spatial correction per slice
GraphBG 0.7216726298458687
Total time in minutes:  2.9700612584749857
